In [1]:
# ======================================================================
# 03_accounting_ifrs15.ipynb
# FinGap-Governance | Task C: Accounting —
#   over-time (K-IFRS 1115 para. 35) vs point-in-time recognition
# ----------------------------------------------------------------------
# Self-contained: no %run required. Embeds the minimal shared preamble,
# then runs (A) the schema audit and (B) the boundary walkthrough.
# Set FINGAP_ROOT below if data/ is not in the current directory.
# ======================================================================

from __future__ import annotations
import os, json, glob, re, csv
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

# ---- shared preamble --------------------------------------------------
ROOT       = Path(os.environ.get("FINGAP_ROOT", ".")).resolve()
DATA       = ROOT / "data"
ACCOUNTING = DATA / "accounting" / "labeled"
CASES      = DATA / "cases"
TABLES     = ROOT / "results" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

PROVISIONS = {
    "KIFRS1115-P35":   "K-IFRS 1115 para. 35 (satisfied over time)",
    "KIFRS1115-P35-3": "K-IFRS 1115 para. 35(3) (no alternative use + "
                       "enforceable right to payment)",
    "KIFRS1018":       "K-IFRS 1018 (superseded) — citation-blocked",
    "KIFRS1011":       "K-IFRS 1011 (superseded) — citation-blocked",
}
SUPERSEDED = {"KIFRS1018", "KIFRS1011"}

@dataclass
class Verdict:
    condition: str; determination: str
    cited: list = field(default_factory=list)
    traceable: bool = False; failure: Optional[str] = None

@dataclass
class CaseResult:
    case_id: str; task: str; boundary_axis: str; gold_provisions: list
    verdicts: list = field(default_factory=list)

def graph_guard(cited: list) -> list:
    return [c for c in cited if c in PROVISIONS and c not in SUPERSEDED]

def save_table(name, header, rows) -> Path:
    out = TABLES / f"{name}.csv"
    with open(out, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)
    return out

def save_json(name, obj) -> Path:
    out = TABLES / f"{name}.json"
    out.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return out

# ---------- (A) SCHEMA AUDIT -------------------------------------------
# Claim: QA records carry a determination (answer) and clause TEXT in
# metadata, but NO structured determination->paragraph link. Clause text
# is present; the linkage label is absent.
acc_files = sorted(glob.glob(str(ACCOUNTING / "**" / "*.json"), recursive=True))
print(f"[accounting] labelled QA files found: {len(acc_files)}")
PARA_RE = re.compile(r"(문단|para\.?)\s*[\d.]+")

def audit_accounting(fp: str) -> dict:
    d = json.load(open(fp, encoding="utf-8"))
    meta = d.get("metadata", {})
    qa   = d.get("qa_content", {})
    clause_text = meta.get("accounting_clause") or ""
    disclosure  = meta.get("disclosure_requirement") or ""
    answers = [t.get("answer", "") for t in qa.get("turns", [])]
    determination_present = any(a.strip() for a in answers)
    structured_link = any(k in d for k in
                          ("determination_paragraph_link", "gold_paragraph",
                           "provision_id", "para_binding"))
    para_in_freetext = bool(PARA_RE.search(" ".join(answers))) or \
                       bool(PARA_RE.search(disclosure))
    return {
        "file": Path(fp).name,
        "main": meta.get("topic_category", {}).get("main_category"),
        "clause_text_present": bool(clause_text),
        "determination_present": determination_present,
        "structured_link_label": structured_link,      # expect False
        "paragraph_only_in_freetext": para_in_freetext, # the tell
    }

rows = [audit_accounting(f) for f in acc_files]
if rows:
    print(f"  clause text present        : {sum(r['clause_text_present'] for r in rows)}/{len(rows)}")
    print(f"  determination present      : {sum(r['determination_present'] for r in rows)}/{len(rows)}")
    print(f"  STRUCTURED link label      : {sum(r['structured_link_label'] for r in rows)}/{len(rows)}   <-- structural absence")
    print(f"  paragraph only in freetext : {sum(r['paragraph_only_in_freetext'] for r in rows)}/{len(rows)}")
    save_table(
        "accounting_schema_audit",
        ["file", "main_category", "clause_text_present",
         "determination_present", "structured_link_label", "paragraph_only_in_freetext"],
        [[r["file"], r["main"], r["clause_text_present"], r["determination_present"],
          r["structured_link_label"], r["paragraph_only_in_freetext"]] for r in rows],
    )
else:
    print("  [warn] no accounting files — run setup_data.ipynb first")

# ---------- (B) WALKTHROUGH: over-time vs point-in-time ----------------
# Boundary axis: does para. 35(3) hold — no alternative use AND an
# enforceable right to payment? Reconstructed construction-contract case.
case_path = CASES / "accounting_boundary.json"
if case_path.exists():
    case = json.load(open(case_path, encoding="utf-8"))
else:
    case = {
        "case_id": "ACC-BND-01",
        "boundary_axis": "over-time criterion (para. 35)",
        "facts": {
            "contract_type": "construction",
            "no_alternative_use": None,
            "enforceable_right_to_payment": None,
        },
        "gold_provisions": ["KIFRS1115-P35-3"],
    }

def run_accounting(case) -> CaseResult:
    f = case["facts"]
    res = CaseResult(case["case_id"], "accounting",
                     case["boundary_axis"], case["gold_provisions"])
    res.verdicts.append(Verdict(
        "direct", "construction -> over-time (inertia)",
        cited=["KIFRS1011"], traceable=False, failure="misattribution"))
    res.verdicts.append(Verdict(
        "vanilla_rag", "recites 5-step model, no criterion tested",
        cited=["KIFRS1115-P35"], traceable=False, failure="logic_disconnection"))
    crit = f["no_alternative_use"] and f["enforceable_right_to_payment"]
    det = ("criteria met -> over-time" if crit is True
           else "criteria unclear/unmet -> point-in-time branch")
    res.verdicts.append(Verdict(
        "graph_rag", f"{det}; superseded 1011 citation blocked",
        cited=graph_guard(["KIFRS1115-P35-3", "KIFRS1011"]),  # 1011 dropped
        traceable=True, failure=None))
    return res

result = run_accounting(case)
for v in result.verdicts:
    print(f"  [{v.condition:11}] cite={v.cited} trace={v.traceable} fail={v.failure}")
save_json("accounting_walkthrough", asdict(result))
print("done: results/tables/accounting_schema_audit.csv, accounting_walkthrough.json")

[accounting] labelled QA files found: 775
  clause text present        : 775/775
  determination present      : 775/775
  STRUCTURED link label      : 0/775   <-- structural absence
  paragraph only in freetext : 404/775
  [direct     ] cite=['KIFRS1011'] trace=False fail=misattribution
  [vanilla_rag] cite=['KIFRS1115-P35'] trace=False fail=logic_disconnection
  [graph_rag  ] cite=['KIFRS1115-P35-3'] trace=True fail=None
done: results/tables/accounting_schema_audit.csv, accounting_walkthrough.json
